# Query a Fabric Data Agent through MCP

Connect directly to the published Fabric Data Agent MCP endpoint, inspect the tools it exposes, and call one of those tools with a natural-language question. The notebook signs you in to the Fabric tenant through the browser, and your account needs access to the Fabric workspace.

## Step 1: Load configuration and create an MCP session helper

Load the Fabric tenant and MCP endpoint written to the project `.env` file during provisioning. The HTTP client follows redirects because Fabric routes MCP requests through its service endpoint.

In [10]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_DATA_AGENT_MCP_URL = os.environ["FABRIC_DATA_AGENT_MCP_URL"]
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)


@asynccontextmanager
async def open_data_agent_session():
    token = credential.get_token(FABRIC_SCOPE)

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        follow_redirects=True,
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(FABRIC_DATA_AGENT_MCP_URL, http_client=http_client,) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print("Fabric Data Agent configuration loaded")

Fabric Data Agent configuration loaded


## Step 2: List the available tools

Initialize an authenticated MCP session and inspect every tool exposed by the published data agent, including each tool's input schema.

In [8]:
from rich.console import Console

console = Console()

async with open_data_agent_session() as session:
    tools_response = await session.list_tools()

data_agent_tools = tools_response.tools

for tool in data_agent_tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

/Users/pamelafox/iqdeepdive-foundryiq-1/.venv/lib/python3.14/site-packages/msal/oauth2cli/oauth2.py:453: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


Tool: DataAgent_ContosoDIYDataAgent

Description: Contoso DIY operations, reviews, and web analytics data agent

Input schema:

{
  "type": "object",
  "properties": {
    "userQuestion": {
      "type": "string"
    }
  },
  "required": [
    "userQuestion"
  ]
}

## Step 3: Query the Fabric ontology

Ask a product inventory question to verify that the data agent routes the request to the Fabric ontology and returns its current inventory quantities.

In [11]:
from rich.markdown import Markdown

question = "Which products currently have the lowest inventory quantities?"

async with open_data_agent_session() as session:
    result = await session.call_tool("DataAgent_ContosoDIYDataAgent", {"userQuestion": question})

answers = [block.text for block in result.content if block.type == "text"]
answer_markdown = "\n\n".join(answers)

deep_link_url = result.structuredContent.get("deepLinkUrl")
answer_markdown += f"\n\n[Open this answer in Fabric]({deep_link_url})"

console.print(Markdown(answer_markdown))

The lowest-inventory products are:                                                                                 

 1 Standard Outlet 15-Amp (ELOTL001) — 90 units on hand; 63 available                                              
 2 Pole Saw Pruner 8-foot (GOPRN002) — 92 on hand; 79 available                                                    
 3 Wall Mount Cabinet (SOCA000017) — 114 on hand; 93 available                                                     
 4 Carriage Bolt Set (HWBO000012) — 119 on hand; 98 available                                                      
 5 Solar Path Light Set (GOOTD001) — 123 on hand; 97 available                                                     

These totals aggregate inventory across their respective inventory records. Complete detail is available in        
lowest_product_inventory.json.                                                                                     

]8;id=10668894;https://msit.fabric.microsoft.com/groups/f200c1bc-e2ec-43b9-9308-be868f68753e/aiskills/eeb78c02-6ae8-4d94-8d93-4476e468650e/stage/published/responses/resp_fabQDz4VXX4OQPZBwJvlqJ2XAYUL/source/mcp\Open this answer in Fabric]8;;\

## Step 4: Query the Power BI semantic model

Ask a website analytics question to verify that the data agent routes the request to the Power BI semantic model and returns its governed revenue and conversion-rate measures.

In [12]:
analytics_question = "What are total website revenue and conversion rate?"

async with open_data_agent_session() as session:
    analytics_result = await session.call_tool(
        "DataAgent_ContosoDIYDataAgent",
        {"userQuestion": analytics_question},
    )

analytics_answers = [
    block.text for block in analytics_result.content if block.type == "text"
]
analytics_answer_markdown = "\n\n".join(analytics_answers)

analytics_deep_link_url = analytics_result.structuredContent.get("deepLinkUrl")
analytics_answer_markdown += (
    f"\n\n[Open this answer in Fabric]({analytics_deep_link_url})"
)

console.print(Markdown(analytics_answer_markdown))

Total website revenue is $126,956.69, and the overall conversion rate is 12.3%. The complete result is available in
website_revenue_conversion_rate.json.                                                                              

]8;id=10668898;https://msit.fabric.microsoft.com/groups/f200c1bc-e2ec-43b9-9308-be868f68753e/aiskills/eeb78c02-6ae8-4d94-8d93-4476e468650e/stage/published/responses/resp_fabOh0lGzfPTqjsxVut33mh8BeYL/source/mcp\Open this answer in Fabric]8;;\